# Demo A: Debussy plays *Children's Corner*

Session 1, *Why encode music?*, part 5. The same demo runs on the course page; here you can change the inputs and keep your work.

Debussy recorded *Children's Corner* himself for the Welte-Mignon reproducing piano (roll 2733). Stanford's SUPRA archive scanned the roll (see it in [Stanford's viewer](https://purl.stanford.edu/dc920rk8870)) and turned the scan into two MIDI files:

- **raw:** every hole in the paper becomes an event, in 4 tracks: bass and treble notes, and two tracks of expression holes that control loudness;
- **expressive:** holes merged into notes, and the expression holes turned into loudness and pedalling.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. Cells marked *Try this* have values you can change; change them and run the cell again.

## Setup

The first cell installs what the notebook needs; run it once after each server start. The second cell defines helper functions; it is collapsed because you do not need to read it.

In [ ]:
%pip install -q verovio==6.3.0 mido matplotlib ipywidgets

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

# Piano roll: time from left to right, pitch from bottom to top, one bar per note.
# The same view CAMAT uses later for its note tables.

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_rgba
from matplotlib.patches import Rectangle

TRACK_COLORS = ["#1f77b4", "#d62728", "#9467bd", "#ff7f0e", "#2ca02c", "#8c564b"]


# notes: [{"pitch", "start", "end"}], optionally "track" (colour) and "alpha".
# bands: [{"start", "end", "row", "color", "label"}] drawn under the keyboard (pedals).
# bar: draw a line every `bar` time units (for scores: quarter notes per bar).
def draw_roll(notes, start=None, end=None, low=None, high=None, bands=(), bar=None,
              unit="s", title=None, width=12, height=None):
    if start is None:
        start = min((n["start"] for n in notes), default=0)
    if end is None:
        end = max((n["end"] for n in notes), default=start + 1)
    pitches = [n["pitch"] for n in notes]
    if low is None:
        low = min(pitches) - 1 if pitches else 59
    if high is None:
        high = max(pitches) + 1 if pitches else 72
    rows = sorted({b["row"] for b in bands})
    below = 2 * len(rows)
    if height is None:
        height = min(8, max(2.5, 0.12 * (high - low + below) + 1))
    fig, ax = plt.subplots(figsize=(width, height))
    for p in range(low, high + 1):
        if p % 12 in (1, 3, 6, 8, 10):
            ax.axhspan(p - 0.5, p + 0.5, color="0.93", lw=0, zorder=0)
    shortest = (end - start) / 1500
    rects = [Rectangle((n["start"], n["pitch"] - 0.4), max(n["end"] - n["start"], shortest), 0.8) for n in notes]
    colors = [to_rgba(TRACK_COLORS[n.get("track", 0) % len(TRACK_COLORS)], n.get("alpha", 0.9)) for n in notes]
    ax.add_collection(PatchCollection(rects, facecolors=colors, edgecolors="none", zorder=2))
    ticks = [(p, f"C{p // 12 - 1}") for p in range(low, high + 1) if p % 12 == 0]
    for b in bands:
        y = low - 1.5 - 2 * rows.index(b["row"])
        ax.add_patch(Rectangle((b["start"], y - 0.6), b["end"] - b["start"], 1.2, color=b["color"], lw=0))
    for row in rows:
        label = next(b["label"] for b in bands if b["row"] == row)
        ticks.append((low - 1.5 - 2 * rows.index(row), label))
    if bar:
        t = bar * (start // bar)
        while t <= end:
            ax.axvline(t, color="0.6", lw=0.8, zorder=1)
            t += bar
    ax.set_yticks([y for y, _ in ticks], [label for _, label in ticks])
    ax.set_xlim(start, end)
    ax.set_ylim(low - 0.5 - below, high + 0.5)
    ax.set_xlabel({"s": "time (seconds)", "q": "time (quarter notes)"}.get(unit, unit))
    ax.set_ylabel("pitch")
    ax.grid(axis="y", color="0.85", lw=0.6)
    if title:
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


# Piano roll of a score read by read_score(): time in quarter notes, a line per bar.
def score_roll(result, **options):
    notes = [{"pitch": n["pitch"], "start": n["on"], "end": n["off"]} for n in result["notes"]]
    meter = result.get("meter") or {"count": 4, "unit": 4}
    draw_roll(notes, start=0, bar=meter["count"] * 4 / meter["unit"], unit="q", **options)

# Read a MIDI file: notes and controllers per track, times in seconds.
# Same rules as the web page's reader: a note-on is paired with the next
# note-off of the same key and channel; notes still open end with the track.

import bisect
import io
import math
import urllib.request

import mido
import numpy as np


def _tempo_map(tempos, ticks_per_beat):
    ticks, seconds, rates = [0], [0.0], [0.5 / ticks_per_beat]  # 120 bpm until the first tempo
    for tick, microseconds in sorted(tempos, key=lambda t: t[0]):
        at = seconds[-1] + (tick - ticks[-1]) * rates[-1]
        rate = microseconds / 1e6 / ticks_per_beat
        if tick == ticks[-1]:
            seconds[-1], rates[-1] = at, rate
        else:
            ticks.append(tick)
            seconds.append(at)
            rates.append(rate)

    def to_seconds(tick):
        i = bisect.bisect_right(ticks, tick) - 1
        return seconds[i] + (tick - ticks[i]) * rates[i]

    return to_seconds


def read_midi(data):
    mid = mido.MidiFile(file=io.BytesIO(data))
    tempos = []
    raw = []
    for track in mid.tracks:
        tick = 0
        events = []
        for msg in track:
            tick += msg.time
            if msg.type == "set_tempo":
                tempos.append((tick, msg.tempo))
            elif msg.type in ("note_on", "note_off", "control_change"):
                events.append((tick, msg))
        raw.append((events, tick))
    to_seconds = _tempo_map(tempos, mid.ticks_per_beat)
    tracks = []
    for index, (events, end_tick) in enumerate(raw):
        open_notes = {}
        notes = []
        controls = []
        for tick, msg in events:
            if msg.type == "control_change":
                controls.append({"control": msg.control, "value": msg.value, "time": to_seconds(tick)})
                continue
            key = (msg.channel, msg.note)
            if msg.type == "note_on" and msg.velocity > 0:
                open_notes.setdefault(key, []).append((tick, msg.velocity))
            elif open_notes.get(key):
                on, velocity = open_notes[key].pop(0)
                notes.append({"pitch": msg.note, "velocity": velocity, "start": to_seconds(on), "end": to_seconds(tick)})
        for (channel, pitch), waiting in open_notes.items():
            for on, velocity in waiting:
                notes.append({"pitch": pitch, "velocity": velocity, "start": to_seconds(on), "end": to_seconds(end_tick)})
        notes.sort(key=lambda n: (n["start"], n["pitch"]))
        tracks.append({"index": index, "notes": notes, "controls": controls})
    return {"tracks": tracks, "duration": max(to_seconds(end) for _, end in raw)}


def load_midi(url):
    with urllib.request.urlopen(url, timeout=60) as response:
        return read_midi(response.read())


# What each track of a SUPRA Welte-Mignon file holds: track -> (label, colour number).
ROLES = {
    "raw": {1: ("bass holes", 0), 2: ("treble holes", 1), 3: ("bass expression holes", 2), 4: ("treble expression holes", 3)},
    "exp": {1: ("bass notes", 0), 2: ("treble notes", 1)},
}


# All notes of one version, with the track number as colour and loudness as opacity.
def roll_notes(midi, version):
    notes = []
    for track in midi["tracks"]:
        if track["index"] not in ROLES[version]:
            continue
        label, colour = ROLES[version][track["index"]]
        for n in track["notes"]:
            alpha = 0.9 if version == "raw" else min(1, 0.3 + (n["velocity"] - 20) / 50)
            notes.append({**n, "track": colour, "role": label, "alpha": alpha})
    return sorted(notes, key=lambda n: n["start"])


# Pedal down (controller value >= 64) to pedal up, as bands under the roll.
def pedal_bands(midi):
    track = next((t for t in midi["tracks"] if any(c["control"] == 64 for c in t["controls"])), None)
    bands = []
    if track is None:
        return bands
    for control, row, color, label in [(64, 0, "#2ca02c", "sustain pedal"), (67, 1, "#8c564b", "soft pedal")]:
        down = None
        for c in track["controls"]:
            if c["control"] != control:
                continue
            if c["value"] >= 64 and down is None:
                down = c["time"]
            if c["value"] < 64 and down is not None:
                bands.append({"start": down, "end": c["time"], "row": row, "color": color, "label": label})
                down = None
    return bands


# A very plain synthesiser: enough to hear rhythm, chords and dynamics, not a piano.
def synth(notes, start, length, rate=22050, dynamics=True):
    out = np.zeros(int(rate * (length + 1)))
    for n in notes:
        if not (start <= n["start"] < start + length and 21 <= n["pitch"] <= 108):
            continue
        dur = max(0.02, min(n["end"], start + length) - n["start"])
        level = 0.04 + 0.2 * max(0, n["velocity"] - 20) / 60 if dynamics else 0.12
        k0 = int((n["start"] - start) * rate)
        k1 = min(len(out), k0 + int((dur + 0.4) * rate))
        t = np.arange(k1 - k0) / rate
        freq = 440 * 2 ** ((n["pitch"] - 69) / 12)
        wave = 2 / math.pi * np.arcsin(np.sin(2 * math.pi * freq * t))  # triangle wave
        env = level * (0.3 + 0.7 * np.exp(-np.maximum(t - 0.01, 0) / 0.4))
        env *= np.minimum(1, t / 0.005)
        env[t > dur] *= np.exp(-(t[t > dur] - dur) / 0.05)
        out[k0:k1] += wave * env
    return out

## 1. Download the two MIDI files

The files come from SUPRA's GitHub repository each time you run this cell (about 330 KB together).

In [ ]:
EXP_URL = "https://raw.githubusercontent.com/pianoroll/SUPRA/master/welte-red/midi-exp/dc920rk8870_exp.mid"
RAW_URL = "https://raw.githubusercontent.com/pianoroll/SUPRA/master/welte-red/midi-raw/dc920rk8870_raw.mid"

rolls = {"raw": load_midi(RAW_URL), "exp": load_midi(EXP_URL)}

for version, midi in rolls.items():
    notes = roll_notes(midi, version)
    kind = "hole events" if version == "raw" else "notes"
    print(f"{version}: {len(notes):,} {kind}, {midi['duration']:.0f} seconds")
    for track in midi["tracks"]:
        if track["index"] in ROLES[version]:
            print(f"   track {track['index']}: {len(track['notes']):6,} {ROLES[version][track['index']][0]}")

## 2. The same seconds in both files

A piano roll shows time from left to right and pitch from bottom to top. In the raw file, the expression holes lie outside the piano's range (below the lowest and above the highest note). In the expressive file, darker means louder, and the bands under the roll show the pedals.

In [ ]:
def show_window(version="exp", start=4, length=20):
    midi = rolls[version]
    notes = [n for n in roll_notes(midi, version) if n["end"] >= start and n["start"] <= start + length]
    if version == "raw":
        draw_roll(notes, start=start, end=start + length, low=14, high=114,
                  title=f"raw: {len(notes):,} hole events in this window")
    else:
        draw_roll(notes, start=start, end=start + length, low=21, high=108, bands=pedal_bands(midi),
                  title=f"expressive: {len(notes):,} notes in this window")


# Try this: another start time (in seconds) or a shorter window.
START = 4
LENGTH = 20
show_window("raw", START, LENGTH)
show_window("exp", START, LENGTH)

## 3. One note, two files

A long note on the roll is punched as a chain of short holes. The expressive file joins the chain into one note. Here is the C3 (MIDI pitch 48) that starts at about 5.1 seconds:

In [ ]:
C3 = 48
for version in ["raw", "exp"]:
    for n in roll_notes(rolls[version], version):
        if n["pitch"] == C3 and 5.0 <= n["start"] <= 5.4:
            print(f"{version}: C3 from {n['start']:.3f} s to {n['end']:.3f} s, {n['end'] - n['start']:.3f} s long")

In [ ]:
show_window("raw", 4.8, 3.6)
show_window("exp", 4.8, 3.6)

## 4. Listen

A very plain synthesiser plays the window: enough to hear rhythm, chords and dynamics, but it is not a piano. The raw version plays only the note holes, all equally loud: the expression holes are not notes.

In [ ]:
from IPython.display import Audio, display

notes_exp = roll_notes(rolls["exp"], "exp")
notes_raw = [n for n in roll_notes(rolls["raw"], "raw") if n["track"] < 2]  # bass and treble holes only

print("Expressive:")
display(Audio(synth(notes_exp, START, LENGTH), rate=22050))
print("Raw:")
display(Audio(synth(notes_raw, START, LENGTH, dynamics=False), rate=22050))

## 5. Explore with sliders

Choose the version, the start and the window length. If you see "Error displaying widget" instead of sliders, reload the browser tab once (the install cell has just added the widget library), or call `show_window(...)` as in part 2.

In [ ]:
import ipywidgets as widgets

widgets.interact(
    show_window,
    version=widgets.Dropdown(options=[("expressive", "exp"), ("raw", "raw")], value="exp"),
    start=widgets.FloatSlider(min=0, max=740, step=0.5, value=4, continuous_update=False),
    length=widgets.Dropdown(options=[5, 10, 20, 40, 80, 160], value=20),
);

## 6. A roll as a table

For a computer, the piano roll is a table: one row per note, with pitch, start, end and loudness. This is the form CAMAT works with later in the course, there with bars and beats from a score instead of seconds.

In [ ]:
print(f"{'pitch':>5}  {'start (s)':>9}  {'end (s)':>8}  {'velocity':>8}  track")
for n in notes_exp[:12]:
    print(f"{n['pitch']:>5}  {n['start']:>9.3f}  {n['end']:>8.3f}  {n['velocity']:>8}  {n['role']}")

## Questions

- Is a roll a score, a recording or a program?
- Where are the bar lines and the quarter notes?
- What is a "note" when 33,508 holes become 6,031 notes?

## Source and licence

MIDI files: SUPRA, Stanford University Piano Roll Archive, roll `dc920rk8870`, licensed [CC BY-NC-SA 4.0](https://github.com/pianoroll/SUPRA). Cite: Zhengshan Shi, Craig Stuart Sapp, Kumaran Arul, Jerry McBride and Julius O. Smith III, "SUPRA: Digitizing the Stanford University Piano Roll Archive", *ISMIR 2019*. The notebook downloads the files when you run it; they are not part of the course materials.